In [ ]:
# ==============================================================================
# CELL 1: Setup & Bronze Ingestion
# ==============================================================================
# Objective: Mount the environment, install dependencies, and ingest the 'offers'
# table from the canonical Bronze Layer database (opus.db).

# --- 1.1 Environment Setup ---
print("Step 1.1: Installing necessary libraries...")
!pip install googlemaps sqlalchemy -q
print("Libraries installed.")

import pandas as pd
from sqlalchemy import create_engine
import googlemaps
import os
from google.colab import drive, userdata
import time

print("\nStep 1.2: Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

# --- 1.2 Configuration & Data Ingress ---
DB_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
TABLE_NAME = 'offers'
print(f"\nStep 1.3: Connecting to Bronze Layer database at '{DB_PATH}'...")
engine = create_engine(f'sqlite:///{DB_PATH}')
offers_df = pd.read_sql_table(TABLE_NAME, engine)
print(f"Successfully ingested {len(offers_df)} records from the '{TABLE_NAME}' table.")

# --- 1.3 API Key Instruction ---
print("\n[ARCHITECT ACTION REQUIRED]: Ensure your Google Maps Platform API key is stored")
print("in Colab Secrets with the name 'GOOGLE_MAPS_API_KEY'.")

Step 1.1: Installing necessary libraries...
Libraries installed.

Step 1.2: Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.

Step 1.3: Connecting to Bronze Layer database at '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'...
Successfully ingested 4766 records from the 'offers' table.

[ARCHITECT ACTION REQUIRED]: Ensure your Google Maps Platform API key is stored
in Colab Secrets with the name 'GOOGLE_MAPS_API_KEY'.


In [ ]:
# ==============================================================================
# CELL 2: Centroid Imputation Engine (REVISED FOR GRANULAR FLAGGING)
# ==============================================================================
# Objective: Impute missing coordinates with granular flags to identify
# whether the pickup, dropoff, or both were imputed.

print("Step 2.1: Initializing Google Maps Client...")
API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=API_KEY)
print("Google Maps client initialized successfully.")

# --- 2.2 Prepare DataFrame for Imputation ---
print("\nStep 2.2: Preparing DataFrame for imputation...")
offers_df['geocoding_quality'] = 'High_Precision'
print(" -> 'geocoding_quality' column created with default 'High_Precision'.")

# --- 2.3 Isolate Imputation Targets ---
pickup_mask = offers_df['pickup_address'].notna() & offers_df['pickup_lat'].isna()
dropoff_mask = offers_df['dropoff_address'].notna() & offers_df['dropoff_lat'].isna()
imputation_targets_mask = pickup_mask | dropoff_mask
imputation_df = offers_df[imputation_targets_mask]
print(f" -> Identified {len(imputation_df)} records requiring coordinate imputation.")

# --- 2.4 Execute Imputation Loop ---
if not imputation_df.empty:
    print("\nStep 2.3: Commencing Granular Centroid Imputation Engine...")
    imputed_pickups = 0
    imputed_dropoffs = 0
    api_errors = 0

    for index, row in imputation_df.iterrows():
        # --- Process PICKUP coordinate if missing ---
        if pd.isna(row['pickup_lat']) and pd.notna(row['pickup_address']):
            try:
                query = f"{row['pickup_address']}, Mexico City, Mexico"
                geocode_result = gmaps.geocode(query)
                if geocode_result:
                    location = geocode_result[0]['geometry']['location']
                    offers_df.at[index, 'pickup_lat'] = location['lat']
                    offers_df.at[index, 'pickup_lon'] = location['lng']
                    # Set the specific pickup flag
                    offers_df.at[index, 'geocoding_quality'] = 'Pickup_Imputed_Centroid'
                    imputed_pickups += 1
                time.sleep(0.05)
            except Exception as e:
                print(f"  -> API Error for offer_id {row.get('offer_id', index)} on pickup: {e}")
                api_errors += 1

        # --- Process DROPOFF coordinate if missing ---
        if pd.isna(row['dropoff_lat']) and pd.notna(row['dropoff_address']):
            try:
                query = f"{row['dropoff_address']}, Mexico City, Mexico"
                geocode_result = gmaps.geocode(query)
                if geocode_result:
                    location = geocode_result[0]['geometry']['location']
                    offers_df.at[index, 'dropoff_lat'] = location['lat']
                    offers_df.at[index, 'dropoff_lon'] = location['lng']

                    # LOGIC UPGRADE: Check if pickup was also imputed for this row
                    if offers_df.at[index, 'geocoding_quality'] == 'Pickup_Imputed_Centroid':
                        offers_df.at[index, 'geocoding_quality'] = 'Both_Imputed_Centroid'
                    else:
                        offers_df.at[index, 'geocoding_quality'] = 'Dropoff_Imputed_Centroid'

                    imputed_dropoffs += 1
                time.sleep(0.05)
            except Exception as e:
                print(f"  -> API Error for offer_id {row.get('offer_id', index)} on dropoff: {e}")
                api_errors += 1

    print("\n--- Imputation Engine Summary ---")
    print(f"Pickup coordinates imputed: {imputed_pickups}")
    print(f"Dropoff coordinates imputed: {imputed_dropoffs}")
    print(f"API / Network errors encountered: {api_errors}")
else:
    print("\nNo records required imputation. Proceeding.")

Step 2.1: Initializing Google Maps Client...
Google Maps client initialized successfully.

Step 2.2: Preparing DataFrame for imputation...
 -> 'geocoding_quality' column created with default 'High_Precision'.
 -> Identified 61 records requiring coordinate imputation.

Step 2.3: Commencing Granular Centroid Imputation Engine...

--- Imputation Engine Summary ---
Pickup coordinates imputed: 55
Dropoff coordinates imputed: 7
API / Network errors encountered: 0


In [ ]:
# ==============================================================================
# CELL 2.5: Review Imputation Results
# ==============================================================================
# Objective: Review and verify the records that have undergone centroid imputation.

print("--- VERIFICATION & REVIEW ---")
review_mask = offers_df['geocoding_quality'] != 'High_Precision'
review_df = offers_df[review_mask]

if not review_df.empty:
    print(f"Displaying the {len(review_df)} records with imputed coordinates for review.")
    display_columns = [
        'offer_id', 'pickup_address', 'pickup_lat', 'pickup_lon',
        'dropoff_address', 'dropoff_lat', 'dropoff_lon', 'geocoding_quality'
    ]
    display(review_df[[col for col in display_columns if col in review_df.columns]])
else:
    print("No records were flagged for imputation. No review is necessary.")

--- VERIFICATION & REVIEW ---
Displaying the 61 records with imputed coordinates for review.


,offer_id,pickup_address,pickup_lat,pickup_lon,dropoff_address,dropoff_lat,dropoff_lon,geocoding_quality
205,OF00206,"Calz De las Águilas, Sur",19.348617,-99.229532,"C. Huemila 11A, San Bartolo Ameyalco, Álvaro O...",19.331080,-99.277646,Pickup_Imputed_Centroid
334,OF00335,"Avenida Desierto de los Leones, Sur",19.335419,-99.252107,"Avenida Revolución 1877, Loreto, 01090 Álvaro ...",19.340141,-99.191895,Pickup_Imputed_Centroid
387,OF00388,Hyattt Campos Eliseos,19.427554,-99.192758,Lagrange 103,19.438406,-99.211107,Dropoff_Imputed_Centroid
414,OF00415,"Avenida Las Águilas, Sur",19.301537,-99.058504,"Contadero, Cuajimalpa de Morelos, 05230 Ciudad...",19.362920,-99.268566,Pickup_Imputed_Centroid
422,OF00423,"Cerrada del Anáhuac, Bosques",19.432608,-99.133208,"Dr. Balmis 148, Doctores, Cuauhtémoc, 06720 Ci...",19.412931,-99.152049,Pickup_Imputed_Centroid
...,...,...,...,...,...,...,...,...
4491,OF04492,Calle Sierra Orizaba & Calle Sierra de Tuxtlah...,19.424938,-99.213085,Calle Colima & Calle Frontera,19.420753,-99.156294,Pickup_Imputed_Centroid
4585,OF04586,"Calle Ribera de Cupia, Bosques",19.398154,-99.238478,"Av Universidad Anáhuac 46, Col Lomas Anáhuac H...",19.402308,-99.262107,Pickup_Imputed_Centroid
4587,OF04588,"Avenida San Antonio, Camino Antiguo a Sante Fe",19.386393,-99.225562,"Moliere 480, Cuajimalpa de Morelos, 05129 Cuaj...",19.392937,-99.263314,Pickup_Imputed_Centroid
4634,OF04635,"Calle Tiro Al Pichón, Bosques",19.391111,-99.245507,"Calle Indiana 255, Col Cd de los Deportes, 038...",19.383277,-99.178840,Pickup_Imputed_Centroid


In [ ]:
# ==============================================================================
# CELL 3: Forge and Save the Silver Layer (REVISED FOR CSV EXPORT)
# ==============================================================================
# Objective: Persist the enriched DataFrame as a canonical Silver Layer artifact
# in CSV format for maximum interoperability.

print("Step 3.1: Defining Silver Layer canonical path...")

# --- 3.1 Configuration ---
SILVER_LAYER_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/03_Analysis_Files'
# MODIFICATION: Changed output filename to reflect the new format.
OUTPUT_FILENAME = 'silver_offers_enriched.csv'
FULL_OUTPUT_PATH = os.path.join(SILVER_LAYER_PATH, OUTPUT_FILENAME)

# --- 3.2 Directory and File Creation ---
print(f" -> Ensuring directory exists: '{SILVER_LAYER_PATH}'")
os.makedirs(SILVER_LAYER_PATH, exist_ok=True)

print(f"\nStep 3.2: Forging Silver Layer artifact as CSV...")
try:
    # MODIFICATION: Switched from to_parquet to to_csv.
    offers_df.to_csv(FULL_OUTPUT_PATH, index=False)

    print("\n--- MISSION COMPLETE ---")
    print("The inaugural Silver Layer artifact has been successfully forged with granular quality flags.")
    print(f"Format: CSV")
    print(f"Location: {FULL_OUTPUT_PATH}")

    # Final verification remains the same
    print("\nVerification - Geocoding Quality Distribution:")
    print(offers_df['geocoding_quality'].value_counts())

except Exception as e:
    raise IOError(f"CRITICAL ERROR: Failed to write the CSV file. Details: {e}")

Step 3.1: Defining Silver Layer canonical path...
 -> Ensuring directory exists: '/content/drive/MyDrive/_Pienza/Assets/Database/03_Analysis_Files'

Step 3.2: Forging Silver Layer artifact as CSV...

--- MISSION COMPLETE ---
The inaugural Silver Layer artifact has been successfully forged with granular quality flags.
Format: CSV
Location: /content/drive/MyDrive/_Pienza/Assets/Database/03_Analysis_Files/silver_offers_enriched.csv

Verification - Geocoding Quality Distribution:
geocoding_quality
High_Precision              4705
Pickup_Imputed_Centroid       54
Dropoff_Imputed_Centroid       6
Both_Imputed_Centroid          1
Name: count, dtype: int64
